In [ ]:
# Author: Sunny Shaban Ali
# Context: Information Retrieval Assignment 2
# References: https://nlp.stanford.edu/IR-book/pdf/irbookonlinereading.pdf, https://www.nltk.org/api/nltk.stem.wordnet.html


import math
import re
import os
from collections import defaultdict, Counter
import nltk

nltk.download('wordnet', quiet=True)

from nltk.stem import WordNetLemmatizer

# Initialize text processing tools
word_lemmatizer = WordNetLemmatizer()

# Read stopwords from a file into a set
def load_stopword_set(stopword_file_path):
    with open(stopword_file_path, 'r') as file:
        return {line.strip() for line in file}


# Read text from file using multiple encoding attempts
def read_file_with_encoding_fallback(file_path):
    supported_encodings = ['utf-8', 'utf-8-sig', 'latin-1']
    for encoding in supported_encodings:
        try:
            with open(file_path, 'r', encoding=encoding) as file:
                return file.read()
        except UnicodeDecodeError:
            continue
    return None

# Convert text to lowercase word tokens
def generate_word_tokens(input_text):
    return re.findall(r'\w+', input_text.lower())

# Clean and lemmatize tokens
def process_and_lemmatize_tokens(word_tokens, stopwords):
    filtered_tokens = []
    for token in word_tokens:
        if token not in stopwords:
            filtered_tokens.append(word_lemmatizer.lemmatize(token))
    return filtered_tokens


# Create a default dictionary for document postings
def create_default_posting_dict():
    return {'document_frequency': 0, 'document_postings': defaultdict(int)}

# Create a default dictionary for document data
def create_default_doc_data():
    return {'term_frequencies': Counter(), 'normalization_value': 0.0}


# Initialize empty index data structures
def create_empty_index_structures():
    inverted_index = defaultdict(create_default_posting_dict)
    forward_index = defaultdict(create_default_doc_data)
    return inverted_index, forward_index

# Construct search indices from document collection
def build_document_search_indices(documents_directory, stopwords):
    inverted_index, forward_index = create_empty_index_structures()
    total_documents = 0

    for filename in os.listdir(documents_directory):
        if not filename.endswith('.txt'):
            continue

        document_id = int(filename.split('.')[0])
        full_file_path = os.path.join(documents_directory, filename)

        file_content = read_file_with_encoding_fallback(full_file_path)
        if file_content is None:
            print(f"Skipping unreadable file: {filename}")
            continue

        # Process document content
        raw_tokens = generate_word_tokens(file_content)
        processed_terms = process_and_lemmatize_tokens(raw_tokens, stopwords)
        term_frequency_counter = Counter(processed_terms)

        # Update forward index
        forward_index[document_id]['term_frequencies'] = term_frequency_counter

        # Update inverted index
        encountered_terms = set()
        for term, frequency in term_frequency_counter.items():
            inverted_index[term]['document_postings'][document_id] = frequency
            if term not in encountered_terms:
                inverted_index[term]['document_frequency'] += 1
                encountered_terms.add(term)

        total_documents += 1

    # Calculate inverse document frequencies
    for term in inverted_index:
        document_frequency = inverted_index[term]['document_frequency']
        inverted_index[term]['idf'] = math.log(total_documents / document_frequency) if document_frequency else 0

    # Calculate document normalization values
    for document_id in forward_index:
        squared_weight_sum = 0.0
        for term, frequency in forward_index[document_id]['term_frequencies'].items():
            term_idf = inverted_index[term]['idf']
            squared_weight_sum += (frequency * term_idf) ** 2
        forward_index[document_id]['normalization_value'] = math.sqrt(squared_weight_sum)

    return inverted_index, forward_index


# Convert search query to processed terms
def prepare_query_terms(search_query, stopwords):
    query_tokens = generate_word_tokens(search_query)
    return process_and_lemmatize_tokens(query_tokens, stopwords)

# Find documents containing query terms
def find_potential_matching_documents(query_terms, inverted_index):
    relevant_documents = set()
    for term in query_terms:
        if term in inverted_index:
            relevant_documents.update(inverted_index[term]['document_postings'].keys())
    return relevant_documents


# Calculate cosine similarity using dot product optimization
def calculate_document_similarity_fast(query_weights, document_id, inverted_index, forward_index):
    dot_product = 0.0
    document_term_freqs = forward_index[document_id]['term_frequencies']
    document_norm = forward_index[document_id]['normalization_value']

    # Compute dot product only for terms that appear in both query and document
    for term, query_weight in query_weights.items():
        if term in document_term_freqs:
            term_freq = document_term_freqs[term]
            term_idf = inverted_index[term]['idf']
            doc_term_weight = term_freq * term_idf
            dot_product += query_weight * doc_term_weight

    # Calculate query norm
    query_norm_squared = 0.0
    for weight in query_weights.values():
        query_norm_squared += weight * weight
    query_norm = math.sqrt(query_norm_squared) or 1.0

    # Compute cosine similarity
    similarity = 0.0
    if document_norm != 0 and query_norm != 0:
        similarity = dot_product / (query_norm * document_norm)

    return similarity


# Sort function for document ID ordering
def sort_by_doc_id(result_item):
    return result_item[0]

# Sort function for descending score ordering
def sort_by_score_desc(result_item):
    return -result_item[1]


#Main query processing workflow
def process_user_search_query(search_query, inverted_index, forward_index, stopwords, similarity_threshold=0.001):
    # Process and weight query terms
    search_terms = prepare_query_terms(search_query, stopwords)
    weighted_query_terms = {}
    for term, count in Counter(search_terms).items():
        if term in inverted_index:
            weighted_query_terms[term] = count * inverted_index[term]['idf']

    # Find candidate documents
    candidate_documents = find_potential_matching_documents(weighted_query_terms.keys(), inverted_index)

    # Score documents using the fast cosine similarity calculation
    ranked_documents = []
    for document_id in candidate_documents:
        document_similarity = calculate_document_similarity_fast(
            weighted_query_terms,
            document_id,
            inverted_index,
            forward_index
        )

        if document_similarity >= similarity_threshold:
            ranked_documents.append((document_id, document_similarity))

    return ranked_documents


# Main application entry point
def main():
    stopword_collection = load_stopword_set('Stopword-List.txt')

    print("Building search indices...")
    search_index, document_index = build_document_search_indices('Abstracts', stopword_collection)

    print("Search system ready. Enter queries below:")
    while True:
        user_query = input("\nSearch query: ").strip()
        if not user_query:
            continue
        if user_query.lower() == 'exit':
            break

        matching_documents = process_user_search_query(user_query, search_index, document_index, stopword_collection)

        # Sort by document ID
        sorted_by_id = sorted(matching_documents, key=sort_by_doc_id)

        # Sort by score (descending)
        sorted_by_score = sorted(matching_documents, key=sort_by_score_desc)

        # Display results sorted by document ID
        print("\nResults sorted by document ID:")
        print("-" * 50)
        print(f"{'Doc ID':<8} {'Score':<15}")
        print("-" * 50)
        for doc_id, score in sorted_by_id:
            print(f"{doc_id:<8} {score:<15.6f}")

        # Display results sorted by relevance score
        print("\nResults sorted by relevance score (descending):")
        print("-" * 50)
        print(f"{'Doc ID':<8} {'Score':<15}")
        print("-" * 50)
        for doc_id, score in sorted_by_score:
            print(f"{doc_id:<8} {score:<15.6f}")

        print(f"\nTotal documents found: {len(matching_documents)}")

if __name__ == '__main__':
    main()